# Budgerigar：单调 codec token-tape 同音复读

有序 tape 保存原始 token，高层理解只能读取、不能改写；网络学习思考间隔、单调读指针和 8 码本原样输出。

In [ ]:
#@title 1. 更新项目
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib
subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train]'],check=True)
sys.path.insert(0,REPO_DIR)
for name in [k for k in list(sys.modules) if k=='budgerigar' or k.startswith('budgerigar.')]:del sys.modules[name]
importlib.invalidate_caches()
print(subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())

In [ ]:
#@title 2. Drive 与 manifest
from google.colab import drive
drive.mount('/content/drive')
WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar')
CODEC_FINGERPRINT='e47d29bb8ba86e3e' #@param {type:'string'}
CODEC_MANIFEST=WORK_ROOT/'manifests'/f'cmu_arctic.encodec.{CODEC_FINGERPRINT}.jsonl'
assert CODEC_MANIFEST.is_file(),CODEC_MANIFEST
import torch,json
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

In [ ]:
#@title 3. 模型形状检查
from budgerigar.codec_copy_model import CodecCopyConfig,create_codec_copy_model
from budgerigar.codec_copy_data import CodecCopyEpisodeDataset,collate_codec_copy
config=CodecCopyConfig(codebooks=8,vocabulary_size=1024,hidden_dim=192,understanding_layers=4)
preview=CodecCopyEpisodeDataset(CODEC_MANIFEST,'train',max_records=2,preload=True)
batch=collate_codec_copy([preview[0],preview[1]])
model=create_codec_copy_model(config)
with torch.no_grad(): logits,voice,diagnostics=model(batch[0])
print('parameters',sum(p.numel() for p in model.parameters()),'logits',logits.shape,'voice',voice.shape)
assert logits.shape[2:]==(8,1024)
assert torch.all(diagnostics['read_phase'][:,1:]>=diagnostics['read_phase'][:,:-1])

In [ ]:
#@title 4. T4 smoke training
MAX_STEPS=200 #@param {type:'integer'}
BATCH_SIZE=2 #@param {type:'integer'}
from budgerigar.train_codec_copy import CodecCopyTrainingConfig,train_codec_copy
RUN_DIR=WORK_ROOT/'checkpoints'/f'codec_copy_{CODEC_FINGERPRINT}'
training=CodecCopyTrainingConfig(batch_size=BATCH_SIZE,max_steps=MAX_STEPS)
report=train_codec_copy(CODEC_MANIFEST,RUN_DIR,training,config)
print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 5. 严格复制门槛
best=max(report['history'],key=lambda x:x['validation_token_accuracy'])
copy_pass=best['validation_token_accuracy']>.99 and best['validation_exact_utterance_rate']>.9 and best['validation_early_voice_rate']<.01
print(json.dumps(best,ensure_ascii=False,indent=2));print('copy_pass =',copy_pass)
if not copy_pass:print('未通过：不进入音色转换。')